# Optimization Mirror Human trên Google Colab

Notebook clone code từ GitHub, cho chọn file upload từ máy, lưu chúng vào Google Drive và ghi kết quả trở lại Drive. Trước khi chạy, chọn **Runtime > Change runtime type > T4 GPU**.

Cấu trúc input mặc định:
```text
MyDrive/optimization_mirror_human/
├── tempupload/
│   ├── input_video.mp4
│   ├── real/
│   │   ├── hmr4d_results.pt
│   │   └── preprocess/vitpose_real.pt
│   └── mirror/
│       ├── hmr4d_results.pt
│       └── preprocess/vitpose_mirror.pt
├── models/                         # chỉ cần nếu model không có trong GitHub/LFS
│   ├── SMPLX_NEUTRAL.npz
│   └── smplx_coco17_J_regressor.pt
└── output/                         # notebook tự tạo thư mục theo thời gian
```

In [ ]:
from google.colab import drive
from datetime import datetime
from pathlib import Path

drive.mount('/content/drive')

# SỬA URL NÀY sau khi đẩy codebase lên GitHub.
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git'
BRANCH = 'main'
DRIVE_ROOT = Path('/content/drive/MyDrive/optimization_mirror_human')
INPUT_DIR = DRIVE_ROOT / 'tempupload'
OUTPUT_DIR = DRIVE_ROOT / 'output' / datetime.now().strftime('%Y%m%d_%H%M%S')
REPO_DIR = Path('/content/optimization_mirror_human')
BYPASS_REFIT = False  # True chỉ để debug nhanh
RENDER_VIDEO = True
MAX_FRAMES = -1       # -1 = toàn bộ video
REUPLOAD = False      # True = chọn lại và ghi đè các file đã có trên Drive


In [ ]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%pip install -q "smplx==0.1.28" "trimesh>=4.0" "rtree>=1.0" "pyrender>=0.1.45"


In [ ]:
from google.colab import files

def upload_one(label, destination, suffix):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and not REUPLOAD:
        print(f'Đã có {label}: {destination}')
        return destination

    print(f'Chọn {label} ({suffix})')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f'{label}: chỉ chọn đúng 1 file')
    filename, content = next(iter(uploaded.items()))
    if Path(filename).suffix.lower() != suffix:
        raise ValueError(f'{label} phải là file {suffix}, nhận được: {filename}')
    destination.write_bytes(content)
    print(f'Đã lưu {label}: {destination}')
    return destination

real_pt = upload_one('HMR4D người thật', INPUT_DIR / 'real' / 'hmr4d_results.pt', '.pt')
mirror_pt = upload_one('HMR4D ảnh gương', INPUT_DIR / 'mirror' / 'hmr4d_results.pt', '.pt')
vitpose_real = upload_one('ViTPose người thật', INPUT_DIR / 'real' / 'preprocess' / 'vitpose_real.pt', '.pt')
vitpose_mirror = upload_one('ViTPose ảnh gương', INPUT_DIR / 'mirror' / 'preprocess' / 'vitpose_mirror.pt', '.pt')
input_video = upload_one('video đầu vào', INPUT_DIR / 'input_video.mp4', '.mp4')
model_path = upload_one('SMPL-X model', DRIVE_ROOT / 'models' / 'SMPLX_NEUTRAL.npz', '.npz')
regressor_path = upload_one('SMPL-X COCO-17 regressor', DRIVE_ROOT / 'models' / 'smplx_coco17_J_regressor.pt', '.pt')


In [ ]:
import os
import sys
import cv2
import torch
import yaml

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required = [real_pt, mirror_pt, vitpose_real, vitpose_mirror, input_video, model_path, regressor_path]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Thiếu file:\n- ' + '\n- '.join(missing))
if model_path.stat().st_size < 10_000_000:
    raise RuntimeError(f'{model_path} không phải file model SMPL-X đầy đủ')
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.')

cap = cv2.VideoCapture(str(input_video))
if not cap.isOpened():
    raise RuntimeError(f'Không mở được video: {input_video}')
image_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_rate = cap.get(cv2.CAP_PROP_FPS) or 30.0
cap.release()

with open(REPO_DIR / 'configs' / 'default.yaml', encoding='utf-8') as file:
    config = yaml.safe_load(file)
config['data']['real_dir'] = str(real_pt)
config['data']['mirror_dir'] = str(mirror_pt)
config['data']['image_width'] = image_width
config['data']['frame_rate'] = frame_rate
config['model']['smpl_model_path'] = str(model_path)
config['visualization']['input_video'] = str(input_video)
config['visualization']['output_dir'] = str(OUTPUT_DIR)
config['visualization']['output_video'] = str(OUTPUT_DIR / 'fused_output.mp4')
config['visualization']['max_frames'] = MAX_FRAMES

with open(OUTPUT_DIR / 'config_used.yaml', 'w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)

print('GPU:', torch.cuda.get_device_name(0))
print('Input:', INPUT_DIR)
print('Output:', OUTPUT_DIR)


In [ ]:
from inference import run_inference

result_pkl = Path(run_inference(config, bypass_refit=BYPASS_REFIT))
print('Đã lưu:', result_pkl)


In [ ]:
if RENDER_VIDEO:
    os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
    from render_video import render_video

    output_video = OUTPUT_DIR / 'fused_output.mp4'
    render_video(
        config,
        str(result_pkl),
        str(input_video),
        str(output_video),
        max_frames=MAX_FRAMES,
    )
    print('Đã lưu:', output_video)


In [ ]:
from IPython.display import Video, display

print('Artifacts:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'- {path.name}: {path.stat().st_size / 1024**2:.1f} MB')
if RENDER_VIDEO and output_video.is_file():
    display(Video(str(output_video), embed=False))
